# GEANT h96 Colab — automated preparation and preflight

This notebook prepares the approved CPU-only GEANT h96 package and performs synthetic tests plus the isolated launcher dry-run. **It does not train by default.** It never includes or requests a test-split artifact. The dataset bundle allowlist is exactly the nine approved train/validation artifacts.

## Before clicking Run all

1. In Colab choose **Runtime → Change runtime type → CPU** and use a Python 3.12 runtime. The first cell stops with the observed version if Python is incompatible; it will not replace Python.
2. Run cells from top to bottom. The dependency cell installs exact pins (NumPy 1.26.4, PyTorch 2.12.0, psutil 7.2.2, pytest 9.1.1). If it reports stale already-imported packages, choose **Runtime → Restart session**, then run all cells again.
3. Choose upload, one explicitly authorized Drive folder, or one existing Colab-local staging folder when prompted. The notebook uses the fixed verified session directory and identifies candidates by SHA-256; `(1)` filename suffixes are accepted only after digest match. It does not recursively scan Drive or `/content`.
4. Continue only if every stage prints PASS and the final dry-run says `preflight_passed_no_training`. Any exception prints its traceback and stops the notebook. A failed/mismatched package must not be repaired by changing hashes in this notebook.
5. The training cell prompts for the exact seed-42/topology authorization phrase after all checks pass. Without it, no training process starts. Preparation and dry-run do not authorize training.
6. After an authorized attempt, that cell packages the run directory and captured launcher log, including failed/incomplete diagnostics, into a unique ZIP and requests a browser download. Save it outside the temporary Colab runtime.

The notebook reuses the exact verified session path when present, and refuses mismatched/partial files rather than overwriting them. Colab runtime behavior is **not verified by local validation**; no training is run locally.

In [ ]:
import hashlib
import importlib.util
import json
import math
import os
import re
import shutil
import signal
import stat
import subprocess
import sys
import traceback
import uuid
import zipfile
from pathlib import Path, PurePosixPath

def stage(name, fn):
    print(f"\n=== {name} ===", flush=True)
    try:
        result = fn()
    except Exception:
        print(f"FAIL: {name}; underlying traceback follows", flush=True)
        traceback.print_exc()
        raise
    print(f"PASS: {name}", flush=True)
    return result

def _check_python():
    observed = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
    print(f"Python: {observed} ({sys.executable})")
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError(f"Incompatible Colab Python {observed}; this package requires Python 3.12. Select a compatible runtime. No packages or data were installed/read by this cell.")
    content_root = Path("/content")
    if not content_root.is_dir() or content_root.is_symlink():
        raise RuntimeError("This notebook expects Google Colab's /content runtime")
    return observed

PYTHON_VERSION = stage("Python and Colab runtime check", _check_python)
SESSION_ID = "e015c2e49eb04dfba84124c39af79a38"
SESSION_ROOT = Path("/content/geant_h96_colab_prepare_v1_e015c2e49eb04dfba84124c39af79a38")
if SESSION_ROOT.is_symlink():
    raise RuntimeError(f"Refusing redirected session root: {SESSION_ROOT}")
SESSION_ROOT.mkdir(mode=0o700, parents=False, exist_ok=True)
INCOMING = SESSION_ROOT / "incoming"
if INCOMING.is_symlink():
    raise RuntimeError(f"Refusing redirected incoming directory: {INCOMING}")
INCOMING.mkdir(mode=0o700, exist_ok=True)
REPOSITORY_ROOT = SESSION_ROOT / "approved_source_snapshot"
TEST_BASETEMP = SESSION_ROOT / f"pytest_v1_{uuid.uuid4().hex}"
TEST_BASETEMP.mkdir(mode=0o700, parents=False, exist_ok=False)
WORKFLOW_ID = uuid.uuid4().hex
SYNTHETIC_SUITE_WORKFLOW_ID = None
DRY_RUN_WORKFLOW_ID = None
print(f"Using verified session workspace: {SESSION_ROOT}")

In [ ]:
PINNED = {"numpy": "1.26.4", "torch": "2.12.0", "psutil": "7.2.2", "pytest": "9.1.1"}

def _install_and_verify_packages():
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError("Refusing package installation outside Python 3.12")
    import importlib.metadata
    module_names = {"numpy": "numpy", "torch": "torch", "psutil": "psutil", "pytest": "pytest"}
    preloaded = {}
    for distribution, module_name in module_names.items():
        module = sys.modules.get(module_name)
        if module is not None:
            preloaded[distribution] = str(getattr(module, "__version__", "unknown"))
    try:
        current = {name: importlib.metadata.version(name) for name in PINNED}
        current["torch"] = current["torch"].split("+", 1)[0]
    except importlib.metadata.PackageNotFoundError:
        current = {}
    if current != PINNED:
        command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check"] + [f"{name}=={version}" for name, version in PINNED.items()]
        print("Installing exact pins:", " ".join(command), flush=True)
        completed = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(completed.stdout, flush=True)
        if completed.returncode != 0:
            raise RuntimeError(f"Pinned dependency installation failed with exit code {completed.returncode}; no fallback versions will be used")
    else:
        print("Exact pinned distributions are already installed; skipping pip to preserve this verified session.", flush=True)
    observed = {name: importlib.metadata.version(name) for name in PINNED}
    observed["torch"] = observed["torch"].split("+", 1)[0]
    mismatches = {name: (PINNED[name], observed[name]) for name in PINNED if observed[name] != PINNED[name]}
    if mismatches:
        raise RuntimeError(f"Installed distributions do not match exact pins: {mismatches}")
    stale = {name: (loaded, PINNED[name]) for name, loaded in preloaded.items() if loaded.split("+", 1)[0] != PINNED[name]}
    if stale:
        raise RuntimeError(f"Packages were already imported at different versions before pip installation: {stale}. Restart the Colab session, then run all cells again; do not continue in this kernel.")
    print("Verified package versions:", json.dumps(observed, sort_keys=True))
    if preloaded:
        print("Already-loaded package versions matched the requested pins:", preloaded)
    return observed

PACKAGE_VERSIONS = stage("Exact dependency installation and verification", _install_and_verify_packages)
PACKAGE_WORKFLOW_ID = WORKFLOW_ID

## Acquisition choice

Choose upload, Drive, or an existing Colab-local folder when prompted. Drive access requires a separate confirmation and an exact folder path. Colab's Drive mount grants broader account-level mount access; this notebook itself only reads matching direct children of the selected folder and never writes to Drive. Use upload or local staging if that mount scope is not acceptable. Only the four approved names (optionally suffixed with `(1)`, `(2)`, etc.) are hashed; no recursive search is performed.

In [ ]:
ACQUISITION_MODE = input("Package source [upload/drive/local] (default upload): ").strip().lower() or "upload"
if ACQUISITION_MODE not in {"upload", "drive", "local"}:
    raise RuntimeError(f"Unsupported package source: {ACQUISITION_MODE!r}")
DRIVE_DIRECTORY = input("Exact Drive folder path (only for drive mode): ").strip() if ACQUISITION_MODE == "drive" else ""
LOCAL_PACKAGE_DIRECTORY = input("Exact existing Colab-local folder (only for local mode): ").strip() if ACQUISITION_MODE == "local" else ""
if ACQUISITION_MODE == "drive":
    if input("Type MOUNT APPROVED DRIVE FOLDER to authorize read-only Drive access: ").strip() != "MOUNT APPROVED DRIVE FOLDER":
        raise RuntimeError("Drive access was not explicitly authorized")

EXPECTED_FILES = {
    "geant_h96_colab_source_v1.zip": "00516e4eb45a2b5310e15d5c1c5fe5553890341db8430a97ec62620074a77f39",
    "source_snapshot_manifest.json": "e42239e0422d16b11e06678a41a4c762314647daa956e5ea587135cebc1d6402",
    "geant_h96_colab_trainval_v1.zip": "df2bac044c9b9a577d5117310a99022cce402b9370e2ba483ea18b8d7045521e",
    "dataset_bundle_manifest.json": "70f60f09b71c0cb89003ac2987af4433df0eeeb64b696845faa25d0fbadb8a95",
}

def _sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _canonical_candidate(actual_name):
    for expected_name in EXPECTED_FILES:
        stem, suffix = os.path.splitext(expected_name)
        if actual_name == expected_name:
            return expected_name
        if re.fullmatch(re.escape(stem) + r" \([1-9][0-9]*\)" + re.escape(suffix), actual_name):
            return expected_name
    return None

def _write_new_bytes(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as stream:
        stream.write(payload)
        stream.flush()
        os.fsync(stream.fileno())

def _safe_components(path):
    absolute = Path(os.path.abspath(path))
    current = Path(absolute.anchor)
    for part in absolute.parts[1:]:
        current = current / part
        info = current.lstat()
        if current.is_symlink() or stat.S_ISLNK(info.st_mode):
            raise RuntimeError(f"Refusing symlink/path redirection component: {current}")
    return absolute

def _acquire_candidates():
    if ACQUISITION_MODE == "upload":
        from google.colab import files
        uploaded = files.upload()
        candidates = []
        for name, payload in uploaded.items():
            canonical = _canonical_candidate(name)
            if canonical is None:
                raise RuntimeError(f"Unexpected uploaded filename {name!r}; upload only the four approved package/manifest files")
            candidates.append((canonical, name, bytes(payload)))
        return candidates
    if ACQUISITION_MODE in {"drive", "local"}:
        if ACQUISITION_MODE == "drive":
            if not DRIVE_DIRECTORY:
                raise RuntimeError("An exact Drive directory is required")
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            source_directory = DRIVE_DIRECTORY
        else:
            if not LOCAL_PACKAGE_DIRECTORY:
                raise RuntimeError("An exact Colab-local package directory is required")
            source_directory = LOCAL_PACKAGE_DIRECTORY
        root = _safe_components(Path(source_directory).expanduser())
        if not root.is_dir():
            raise RuntimeError(f"Authorized Drive directory does not exist or is not a directory: {root}")
        candidates = []
        for path in root.iterdir():
            canonical = _canonical_candidate(path.name)
            if canonical is None:
                continue
            info = path.lstat()
            if not stat.S_ISREG(info.st_mode) or path.is_symlink():
                raise RuntimeError(f"Expected an ordinary file, not a redirected/non-file path: {path}")
            candidates.append((canonical, path.name, path))
        return candidates
    raise RuntimeError(f"Unknown acquisition mode: {ACQUISITION_MODE!r}")

def _identify_and_stage():
    _safe_components(INCOMING)
    candidates = _acquire_candidates()
    matches = {name: [] for name in EXPECTED_FILES}
    for canonical, observed_name, payload_or_path in candidates:
        digest = _sha256_bytes(payload_or_path) if isinstance(payload_or_path, bytes) else _sha256_file(payload_or_path)
        expected = EXPECTED_FILES[canonical]
        print(f"Candidate {observed_name!r}: SHA-256 {digest}; expected {expected}")
        if digest != expected:
            raise RuntimeError(f"Digest mismatch for candidate {observed_name!r}; refusing substitutions or older bundle versions")
        matches[canonical].append((observed_name, payload_or_path))
    missing = [name for name, values in matches.items() if len(values) == 0]
    duplicates = [name for name, values in matches.items() if len(values) > 1]
    if missing or duplicates:
        raise RuntimeError(f"Need exactly one digest-matched file for each of four artifacts; missing={missing}, duplicate_matches={duplicates}")
    staged = {}
    for canonical, values in matches.items():
        payload_or_path = values[0][1]
        destination = INCOMING / canonical
        if destination.exists():
            if destination.is_symlink() or _sha256_file(destination) != EXPECTED_FILES[canonical]:
                raise RuntimeError(f"Refusing existing mismatched staged file: {destination}")
            staged[canonical] = destination
            continue
        if isinstance(payload_or_path, bytes):
            _write_new_bytes(destination, payload_or_path)
        else:
            with payload_or_path.open("rb") as source, destination.open("xb") as output:
                shutil.copyfileobj(source, output, 1024 * 1024)
        if _sha256_file(destination) != EXPECTED_FILES[canonical]:
            raise RuntimeError(f"Staged copy failed digest verification: {canonical}")
        staged[canonical] = destination
    print("Exactly four files identified by SHA-256; suffixes such as (1) are accepted only after digest match.")
    return staged

STAGED_FILES = stage("Acquire and identify approved packages", _identify_and_stage)

In [ ]:
EXPECTED_DATA_ARTIFACTS = (
    "metadata.json",
    "timestamps_train.npy",
    "timestamps_validation.npy",
    "utilization_train.npy",
    "utilization_validation.npy",
    "valid_train.npy",
    "valid_validation.npy",
    "window_index_train_h96.json",
    "window_index_validation_h96.json",
)
SOURCE_INTERNAL_MANIFEST = "PORTABLE_SOURCE_MANIFEST.json"
REQUIRED_SOURCE_PATHS = {
    "phase9_geant/colab_h96_v1/launch.py",
    "phase9_geant/colab_h96_v1/package.py",
    "phase9_geant/colab_h96_v1/linux_resources.py",
    "phase9_geant/colab_h96_v1/environment.py",
    "phase9_geant/horizon96_streaming_experiment_v2.py",
    "phase9_geant/split_access.py",
    "phase9_geant/tests/test_horizon96_streaming_experiment.py",
    "phase9_geant/tests/test_horizon96_streaming_experiment_v2.py",
    "phase9_geant/tests/test_spatiotemporal_link.py",
    "phase9_geant/colab_h96_v1/tests/test_portability.py",
    "data/raw/geant_topology/geant.txt",
    "requirements-colab-h96.txt",
}

def _safe_member(name):
    if not isinstance(name, str) or not name or "\\" in name or "\x00" in name:
        raise RuntimeError(f"Unsafe archive member name: {name!r}")
    path = PurePosixPath(name)
    if path.is_absolute() or not path.parts or ":" in path.parts[0] or any(part in ("", ".", "..") for part in path.parts):
        raise RuntimeError(f"Unsafe archive member path: {name!r}")
    return path

def _zip_inventory(zf):
    infos = zf.infolist()
    names = []
    folded = set()
    for info in infos:
        _safe_member(info.filename)
        if info.is_dir():
            raise RuntimeError(f"Explicit directory entries are not permitted: {info.filename}")
        mode = (info.external_attr >> 16) & 0xffff
        kind = stat.S_IFMT(mode)
        if stat.S_ISLNK(mode) or (kind not in (0, stat.S_IFREG)):
            raise RuntimeError(f"Non-regular ZIP member prohibited: {info.filename}")
        key = info.filename.casefold()
        if key in folded:
            raise RuntimeError(f"Duplicate or case-colliding ZIP member: {info.filename}")
        folded.add(key)
        names.append(info.filename)
    return names

def _hash_zip_member(zf, name):
    digest = hashlib.sha256()
    count = 0
    with zf.open(name, "r") as source:
        while True:
            block = source.read(1024 * 1024)
            if not block:
                break
            count += len(block)
            digest.update(block)
    return count, digest.hexdigest()

def _verify_bundle(kind, archive_path, manifest_path, expected_archive_hash, expected_manifest_hash):
    if _sha256_file(archive_path) != expected_archive_hash:
        raise RuntimeError(f"{kind} archive digest mismatch")
    manifest_bytes = manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != expected_manifest_hash:
        raise RuntimeError(f"{kind} external manifest digest mismatch")
    manifest = json.loads(manifest_bytes.decode("utf-8"))
    if manifest.get("archive_sha256") != expected_archive_hash:
        raise RuntimeError(f"{kind} manifest points to a different archive digest")
    with zipfile.ZipFile(archive_path, "r") as zf:
        names = _zip_inventory(zf)
        if kind == "source":
            if manifest.get("schema") != "geant-h96-colab-source-snapshot-v1":
                raise RuntimeError("Unsupported source manifest schema")
            entries_list = manifest.get("entries")
            if not isinstance(entries_list, list):
                raise RuntimeError("Source manifest entries are missing")
            entries = {}
            folded = set()
            for entry in entries_list:
                name = entry.get("path")
                _safe_member(name)
                if name.casefold() in folded:
                    raise RuntimeError(f"Duplicate source manifest path: {name}")
                folded.add(name.casefold())
                if not isinstance(entry.get("size_bytes"), int) or entry["size_bytes"] < 0:
                    raise RuntimeError(f"Invalid source size for {name}")
                digest = entry.get("sha256")
                if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):
                    raise RuntimeError(f"Invalid source SHA-256 for {name}")
                entries[name] = entry
            if not REQUIRED_SOURCE_PATHS.issubset(entries):
                raise RuntimeError(f"Approved source inventory is missing required paths: {sorted(REQUIRED_SOURCE_PATHS-set(entries))}")
            if set(names) != set(entries) | {SOURCE_INTERNAL_MANIFEST} or len(names) != len(entries) + 1:
                raise RuntimeError("Source ZIP exact inventory differs from its approved external manifest")
            internal_bytes = zf.read(SOURCE_INTERNAL_MANIFEST)
            internal = json.loads(internal_bytes.decode("utf-8"))
            if (internal.get("entries") != entries_list
                    or internal.get("first_party_import_inventory") != manifest.get("first_party_import_inventory")
                    or internal.get("schema") != manifest.get("schema")):
                raise RuntimeError("Internal/external source manifests disagree")
            expected_imports = {e["path"]: e.get("first_party_imports", []) for e in entries_list if e["path"].endswith(".py")}
            if manifest.get("first_party_import_inventory") != expected_imports:
                raise RuntimeError("Source import inventory is inconsistent")
        elif kind == "dataset":
            if manifest.get("schema") != "geant-h96-colab-trainval-bundle-v1":
                raise RuntimeError("Unsupported dataset manifest schema")
            artifacts = manifest.get("artifacts")
            if not isinstance(artifacts, list) or [item.get("path") for item in artifacts] != list(EXPECTED_DATA_ARTIFACTS):
                raise RuntimeError("Dataset manifest does not list the exact nine train/validation artifacts in order")
            if manifest.get("exact_archive_members") != list(EXPECTED_DATA_ARTIFACTS):
                raise RuntimeError("Dataset exact member allowlist differs")
            if set(names) != set(EXPECTED_DATA_ARTIFACTS) or len(names) != len(EXPECTED_DATA_ARTIFACTS):
                raise RuntimeError("Dataset ZIP must contain exactly the nine approved files")
            if manifest.get("source_scope") != {"train": [0, 8020], "validation": [8020, 9740]}:
                raise RuntimeError("Dataset split boundaries differ from the approved protocol")
            if manifest.get("history_length") != 672 or manifest.get("horizon") != 96:
                raise RuntimeError("Dataset manifest is not the approved h96/672 protocol")
            if manifest.get("window_counts") != {"train": 3924, "validation": 953}:
                raise RuntimeError("Dataset window counts differ from the reviewed package")
            topology = manifest.get("topology", {})
            if (topology.get("node_count") != 22 or topology.get("undirected_link_count") != 36
                    or topology.get("directed_link_count") != 72):
                raise RuntimeError("Dataset topology structural contract mismatch")
            entries = {item["path"]: item for item in artifacts}
            if any("test" in name.casefold() or "h4" in name.casefold() for name in names):
                raise RuntimeError("Dataset bundle includes a forbidden test/h4 member")
        else:
            raise RuntimeError(f"Unknown archive kind {kind!r}")
        for name, entry in entries.items():
            size, digest = _hash_zip_member(zf, name)
            if size != entry.get("size_bytes") or digest != entry.get("sha256"):
                raise RuntimeError(f"Archive member size/hash mismatch: {name}")
        if kind == "source":
            _, internal_hash = _hash_zip_member(zf, SOURCE_INTERNAL_MANIFEST)
            if len(internal_hash) != 64:
                raise RuntimeError("Invalid internal source manifest hash")
    return manifest

def _verify_packages():
    expected = EXPECTED_FILES
    source_archive = STAGED_FILES["geant_h96_colab_source_v1.zip"]
    source_manifest = STAGED_FILES["source_snapshot_manifest.json"]
    dataset_archive = STAGED_FILES["geant_h96_colab_trainval_v1.zip"]
    dataset_manifest = STAGED_FILES["dataset_bundle_manifest.json"]
    source_info = _verify_bundle("source", source_archive, source_manifest, expected[source_archive.name], expected[source_manifest.name])
    dataset_info = _verify_bundle("dataset", dataset_archive, dataset_manifest, expected[dataset_archive.name], expected[dataset_manifest.name])
    print(f"Source entries verified: {len(source_info['entries'])}; dataset artifacts: {len(dataset_info['artifacts'])}")
    return source_info, dataset_info

VERIFIED_MANIFESTS = stage("Independent archive, manifest, path, hash and inventory verification", _verify_packages)
SOURCE_MANIFEST, DATASET_MANIFEST = VERIFIED_MANIFESTS

In [ ]:
def _verify_existing_extraction(source_manifest, dataset_manifest):
    if not REPOSITORY_ROOT.is_dir() or REPOSITORY_ROOT.is_symlink():
        raise RuntimeError(f"Existing source snapshot is not a regular directory: {REPOSITORY_ROOT}")
    with zipfile.ZipFile(STAGED_FILES["geant_h96_colab_source_v1.zip"], "r") as zf:
        internal_bytes = zf.read(SOURCE_INTERNAL_MANIFEST)
    source_expected = {item["path"]: item for item in source_manifest["entries"]}
    source_expected[SOURCE_INTERNAL_MANIFEST] = {"size_bytes": len(internal_bytes), "sha256": hashlib.sha256(internal_bytes).hexdigest()}
    for name, entry in source_expected.items():
        path = REPOSITORY_ROOT.joinpath(*_safe_member(name).parts)
        _safe_components(path)
        if not path.is_file() or path.stat().st_size != entry["size_bytes"] or _sha256_file(path) != entry["sha256"]:
            raise RuntimeError(f"Existing extracted source differs from approved bundle: {name}")
    data_root = REPOSITORY_ROOT / "data/processed/geant_horizon96_streaming_v1"
    _safe_components(data_root)
    if not data_root.is_dir() or data_root.is_symlink():
        raise RuntimeError(f"Existing extracted dataset root is missing or redirected: {data_root}")
    if {item.name for item in data_root.iterdir()} != set(EXPECTED_DATA_ARTIFACTS):
        raise RuntimeError("Existing dataset extraction has an unexpected inventory")
    data_expected = {item["path"]: item for item in dataset_manifest["artifacts"]}
    for name, entry in data_expected.items():
        path = data_root / name
        _safe_components(path)
        if not path.is_file() or path.stat().st_size != entry["size_bytes"] or _sha256_file(path) != entry["sha256"]:
            raise RuntimeError(f"Existing extracted dataset differs from approved bundle: {name}")
    expected_tree_files = set(source_expected) | {"data/processed/geant_horizon96_streaming_v1/" + name for name in EXPECTED_DATA_ARTIFACTS}
    actual_tree_files = set()
    for item in REPOSITORY_ROOT.rglob("*"):
        _safe_components(item)
        if item.is_file():
            actual_tree_files.add(item.relative_to(REPOSITORY_ROOT).as_posix())
    if actual_tree_files != expected_tree_files:
        raise RuntimeError(f"Approved session source tree has unexpected/missing files: extra={sorted(actual_tree_files - expected_tree_files)}, missing={sorted(expected_tree_files - actual_tree_files)}")
    return data_root

def _extract_verified():
    source_manifest = SOURCE_MANIFEST
    dataset_manifest = DATASET_MANIFEST
    if REPOSITORY_ROOT.exists():
        data_root = _verify_existing_extraction(source_manifest, dataset_manifest)
        print("Reusing existing extracted project only after source/data hashes and exact data inventory matched.")
        return data_root
    # The source archive is authenticated by the externally pinned digest above. Recheck
    # archive/member hashes while streaming each member into a fresh root.
    REPOSITORY_ROOT.mkdir(mode=0o700, parents=False, exist_ok=False)
    with zipfile.ZipFile(STAGED_FILES["geant_h96_colab_source_v1.zip"], "r") as zf:
        for name in sorted(set(item["path"] for item in source_manifest["entries"]) | {SOURCE_INTERNAL_MANIFEST}):
            relative = _safe_member(name)
            target = REPOSITORY_ROOT.joinpath(*relative.parts)
            target.parent.mkdir(mode=0o700, parents=True, exist_ok=True)
            parent = target.parent.resolve(strict=True)
            root_real = REPOSITORY_ROOT.resolve(strict=True)
            if parent != Path(os.path.abspath(target.parent)) or (parent != root_real and root_real not in parent.parents):
                raise RuntimeError(f"Extraction containment check failed: {name}")
            entry = next((item for item in source_manifest["entries"] if item["path"] == name), None)
            expected_size = entry["size_bytes"] if entry else zf.getinfo(name).file_size
            expected_hash = entry["sha256"] if entry else hashlib.sha256(zf.read(name)).hexdigest()
            digest = hashlib.sha256()
            count = 0
            with zf.open(name, "r") as source, target.open("xb") as output:
                while True:
                    block = source.read(1024 * 1024)
                    if not block:
                        break
                    count += len(block)
                    digest.update(block)
                    output.write(block)
            if count != expected_size or digest.hexdigest() != expected_hash:
                raise RuntimeError(f"Extracted member differs from verified digest: {name}")
    data_root = REPOSITORY_ROOT / "data/processed/geant_horizon96_streaming_v1"
    if data_root.exists():
        raise FileExistsError(f"Refusing existing extracted dataset root: {data_root}")
    # Independent exact allowlist extraction; the package API below repeats this check.
    data_root.mkdir(mode=0o700, parents=True, exist_ok=False)
    _safe_components(data_root)
    if data_root.resolve(strict=True) != Path(os.path.abspath(data_root)):
        raise RuntimeError("Dataset extraction root redirected after creation")
    with zipfile.ZipFile(STAGED_FILES["geant_h96_colab_trainval_v1.zip"], "r") as zf:
        for name in EXPECTED_DATA_ARTIFACTS:
            target = data_root / name
            with zf.open(name, "r") as source, target.open("xb") as output:
                shutil.copyfileobj(source, output, 1024 * 1024)
    actual = {item.name for item in data_root.iterdir()}
    if actual != set(EXPECTED_DATA_ARTIFACTS):
        raise RuntimeError(f"Extracted dataset inventory mismatch: {sorted(actual)}")
    _verify_existing_extraction(source_manifest, dataset_manifest)
    return data_root

DATA_ROOT = stage("Fresh safe extraction and exact dataset inventory", _extract_verified)
def _load_verified_portable_package():
    package_path = _safe_components(REPOSITORY_ROOT / "phase9_geant/colab_h96_v1/package.py")
    manifest_entry = next((entry for entry in SOURCE_MANIFEST["entries"] if entry["path"] == "phase9_geant/colab_h96_v1/package.py"), None)
    if manifest_entry is None or _sha256_file(package_path) != manifest_entry.get("sha256"):
        raise RuntimeError("Portable verifier source is absent or differs from the verified snapshot manifest")
    module_name = "_approved_colab_package_" + manifest_entry["sha256"]
    spec = importlib.util.spec_from_file_location(module_name, package_path)
    if spec is None or spec.loader is None:
        raise RuntimeError("Unable to load package verifier from the approved snapshot path")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    try:
        spec.loader.exec_module(module)
    except BaseException:
        sys.modules.pop(module_name, None)
        raise
    return module

portable_package = _load_verified_portable_package()

def _package_reverification():
    portable_package.verify_source_bundle(
        STAGED_FILES["geant_h96_colab_source_v1.zip"], STAGED_FILES["source_snapshot_manifest.json"],
        EXPECTED_FILES["geant_h96_colab_source_v1.zip"], EXPECTED_FILES["source_snapshot_manifest.json"],
    )
    portable_package.verify_dataset_bundle(
        STAGED_FILES["geant_h96_colab_trainval_v1.zip"], STAGED_FILES["dataset_bundle_manifest.json"],
        EXPECTED_FILES["geant_h96_colab_trainval_v1.zip"], EXPECTED_FILES["dataset_bundle_manifest.json"],
    )
    portable_package.verify_extracted_source(REPOSITORY_ROOT, SOURCE_MANIFEST)
    metadata = portable_package._read_dataset_metadata(DATA_ROOT)
    if metadata["protocol_id"] != "geant-horizon96-streaming-trainval-v1":
        raise RuntimeError("Extracted dataset protocol identifier mismatch")
    topology = metadata.get("topology", {})
    bundle_topology = DATASET_MANIFEST.get("topology", {})
    if (topology.get("node_count") != 22 or topology.get("undirected_link_count") != 36
            or topology.get("directed_link_count") != 72
            or len(topology.get("directed_edges_interleaved", ())) != 72
            or topology.get("ordered_edge_sha256") != bundle_topology.get("ordered_edge_sha256")):
        raise RuntimeError("Ordered directed-edge identity differs between dataset metadata and approved bundle manifest")
    print("Dataset split and window contract:", metadata["source_scope"], metadata["windows"])
    return metadata

DATASET_METADATA = stage("Portable package deep verification", _package_reverification)
print("Verified dataset protocol:", DATASET_METADATA["protocol_id"])

In [ ]:
SYNTHETIC_TESTS = [
    "phase9_geant/colab_h96_v1/tests/test_portability.py",
    "phase9_geant/tests/test_horizon96_streaming_experiment.py",
    "phase9_geant/tests/test_horizon96_streaming_experiment_v2.py",
    "phase9_geant/tests/test_spatiotemporal_link.py",
]

def _run_synthetic_suite():
    for relative in SYNTHETIC_TESTS:
        if not (REPOSITORY_ROOT / relative).is_file():
            raise FileNotFoundError(f"Required synthetic test is absent from approved source bundle: {relative}")
    if TEST_BASETEMP.exists() and any(TEST_BASETEMP.iterdir()):
        raise FileExistsError(f"Refusing nonempty pytest basetemp: {TEST_BASETEMP}")
    command = [sys.executable, "-m", "pytest", "-q", "-rs", "--basetemp", str(TEST_BASETEMP), *SYNTHETIC_TESTS]
    print("Command:", " ".join(command), flush=True)
    result = subprocess.run(command, cwd=REPOSITORY_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout, flush=True)
    if result.returncode != 0:
        raise RuntimeError(f"Approved four-file synthetic suite failed with exit code {result.returncode}")
    summary_lines = [line.strip() for line in result.stdout.splitlines() if re.search(r"\d+ passed\b|\d+ failed\b|\d+ skipped\b|\d+ error\b", line)]
    passed_match = re.search(r"\b([0-9]+) passed\b", summary_lines[-1]) if summary_lines else None
    if passed_match is None or int(passed_match.group(1)) != 68:
        raise RuntimeError(f"Expected exactly 68 passing synthetic tests, got: {summary_lines[-1] if summary_lines else 'no pytest summary'}")
    if re.search(r"\b[1-9][0-9]* (?:failed|skipped|error|xfailed|xpassed)\b", summary_lines[-1]):
        raise RuntimeError(f"Synthetic suite must have zero failures and zero skips: {summary_lines[-1]}")
    print("Pytest summary:", summary_lines[-1], flush=True)
    return {"exit_code": result.returncode, "test_files": SYNTHETIC_TESTS, "basetemp": str(TEST_BASETEMP)}

SYNTHETIC_SUITE_RESULT = stage("Complete synthetic test suite", _run_synthetic_suite)
SYNTHETIC_SUITE_WORKFLOW_ID = WORKFLOW_ID

In [ ]:
def _run_launcher_dry_run():
    expected_output_root = REPOSITORY_ROOT / "data/processed/geant_horizon96_streaming_experiment_v2_colab_memv3"
    if expected_output_root.exists():
        raise FileExistsError(f"Refusing pre-existing experiment namespace: {expected_output_root}")
    launcher = REPOSITORY_ROOT / "phase9_geant/colab_h96_v1/launch.py"
    command = [
        sys.executable, "-I", "-S", str(launcher), "--dry-run",
        "--repository-root", str(REPOSITORY_ROOT),
        "--source-bundle", str(STAGED_FILES["geant_h96_colab_source_v1.zip"]),
        "--source-manifest", str(STAGED_FILES["source_snapshot_manifest.json"]),
        "--expected-source-bundle-sha256", EXPECTED_FILES["geant_h96_colab_source_v1.zip"],
        "--expected-source-manifest-sha256", EXPECTED_FILES["source_snapshot_manifest.json"],
        "--dataset-bundle", str(STAGED_FILES["geant_h96_colab_trainval_v1.zip"]),
        "--dataset-manifest", str(STAGED_FILES["dataset_bundle_manifest.json"]),
        "--expected-dataset-bundle-sha256", EXPECTED_FILES["geant_h96_colab_trainval_v1.zip"],
        "--expected-dataset-manifest-sha256", EXPECTED_FILES["dataset_bundle_manifest.json"],
        "--seed", "42", "--adjacency-mode", "topology",
    ]
    print("Command:", " ".join(command), flush=True)
    result = subprocess.run(command, cwd=REPOSITORY_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print(result.stdout, flush=True)
    if result.stderr:
        print("Launcher stderr:", result.stderr, flush=True)
    if result.returncode != 0:
        if result.stderr:
            print("Underlying launcher failure output:", result.stderr, flush=True)
        raise RuntimeError(f"Isolated launcher dry-run failed with exit code {result.returncode}")
    summary = json.loads(result.stdout)
    if summary.get("status") != "preflight_passed_no_training":
        raise RuntimeError(f"Unexpected dry-run status: {summary.get('status')!r}")
    if summary.get("seed") != 42 or summary.get("adjacency_mode") != "topology" or summary.get("horizon") != 96 or summary.get("history_length") != 672:
        raise RuntimeError("Dry-run experiment identity differs from approved seed-42 h96 topology scope")
    expected_run = REPOSITORY_ROOT / "data/processed/geant_horizon96_streaming_experiment_v2_colab_memv3/h96_topology_seed_42"
    if Path(summary.get("run_destination", "")) != expected_run or Path(summary.get("output_root", "")) != expected_run.parent:
        raise RuntimeError("Dry-run returned an unexpected or redirected run namespace")
    if summary.get("data_protocol_id") != "geant-horizon96-streaming-trainval-v1":
        raise RuntimeError("Dry-run data protocol identity mismatch")
    if summary.get("environment", {}).get("device_policy") != "CPU only for this migration protocol":
        raise RuntimeError("Dry-run environment is not configured for CPU-only execution")
    resource = summary.get("linux_resource_preflight", {})
    policy = resource.get("policy", {})
    expected_policy = "colab-linux-process-memory-v3-visible-cgroup"
    startup_headroom = 2 * 1024**3 + 384 * 1024**2
    if resource.get("status") != "pass":
        raise RuntimeError(f"Linux resource preflight did not pass: {resource}")
    if policy.get("policy_id") != expected_policy or policy.get("process_peak_limit_bytes") != 2 * 1024**3:
        raise RuntimeError(f"Colab memory-policy identity/limit mismatch: {policy}")
    if (policy.get("startup_headroom_bytes") != startup_headroom
            or policy.get("minimum_free_disk_bytes") != 512 * 1024**2
            or policy.get("in_run_os_available_reserve_bytes") != 384 * 1024**2
            or policy.get("in_run_cgroup_headroom_reserve_bytes") != 384 * 1024**2):
        raise RuntimeError(f"Colab memory/disk reserve mismatch: {policy}")
    if policy.get("visible_unlimited_cgroup_policy") != "allow literal unlimited visible limits only; host and process guards remain mandatory":
        raise RuntimeError("Visible-cgroup policy statement differs from reviewed v3")
    if summary.get("train_windows") != 3924 or summary.get("validation_windows") != 953:
        raise RuntimeError("Dry-run train/validation window counts differ from the approved dataset")
    if summary.get("test_access") != "no test files included or loaded":
        raise RuntimeError("Dry-run did not affirm the train/validation-only access contract")
    if summary.get("source_bundle_sha256") != EXPECTED_FILES["geant_h96_colab_source_v1.zip"] or summary.get("source_snapshot_manifest_sha256") != EXPECTED_FILES["source_snapshot_manifest.json"]:
        raise RuntimeError("Dry-run source bundle identity does not match the pinned values")
    if summary.get("dataset_bundle_sha256") != EXPECTED_FILES["geant_h96_colab_trainval_v1.zip"] or summary.get("dataset_manifest_sha256") != EXPECTED_FILES["dataset_bundle_manifest.json"]:
        raise RuntimeError("Dry-run dataset bundle identity does not match the pinned values")
    if expected_output_root.exists() or expected_run.exists():
        raise RuntimeError("Dry-run unexpectedly created an experiment result namespace")
    print("PASS: source and dataset isolated; training not launched; namespace remains unused.", flush=True)
    return summary

DRY_RUN_RESULT = stage("Isolated launcher dry-run (no training)", _run_launcher_dry_run)
DRY_RUN_WORKFLOW_ID = WORKFLOW_ID
OUTPUT_ROOT = REPOSITORY_ROOT / "data/processed/geant_horizon96_streaming_experiment_v2_colab_memv3"
RUN_DIR = OUTPUT_ROOT / "h96_topology_seed_42"

## One seed-42 training attempt (explicit authorization required)

After all preparation, synthetic tests, bundle checks, and the isolated v3 resource dry-run pass, this cell asks for the exact phrase shown. Do not enter it unless you separately authorize this single seed-42 topology attempt. Existing output is never resumed or overwritten. This attempt exports a unique archive automatically, including failed-run diagnostics.

In [ ]:
AUTHORIZATION_PHRASE = "AUTHORIZE SEED 42 TOPOLOGY TRAINING"

def _archive_attempt(run_dir, log_path, session_root, download_fn=None):
    archive_path = session_root / f"seed42_topology_attempt_{uuid.uuid4().hex}.zip"
    with zipfile.ZipFile(archive_path.open("xb"), "w", compression=zipfile.ZIP_DEFLATED) as archive:
        if log_path is not None and log_path.is_file():
            archive.write(log_path, arcname="launcher_output.log")
        if run_dir.is_dir():
            root = _safe_components(run_dir)
            for item in sorted(root.rglob("*")):
                _safe_components(item)
                if item.is_symlink():
                    raise RuntimeError(f"Refusing symlink in run artifacts: {item}")
                if item.is_dir():
                    continue
                if not item.is_file():
                    raise RuntimeError(f"Refusing non-regular run artifact: {item}")
                relative = item.relative_to(root).as_posix()
                if re.search(r"(^|[_.-])test([_.-]|$)", PurePosixPath(relative).name, re.I):
                    raise RuntimeError(f"Refusing test-named artifact in run export: {relative}")
                archive.write(item, arcname=f"run/{relative}")
    digest = _sha256_file(archive_path)
    print(f"Attempt export created: {archive_path}; SHA-256={digest}", flush=True)
    try:
        if download_fn is None:
            from google.colab import files
            download_fn = files.download
        download_fn(str(archive_path))
        print("Browser download requested; save this ZIP outside the temporary Colab runtime.", flush=True)
    except Exception as exc:
        print(f"Download failed; archive remains at {archive_path}: {type(exc).__name__}: {exc}", flush=True)
    return {"path": str(archive_path), "sha256": digest}

def _authorized_training_workflow(prompt_fn=None, popen_factory=None, download_fn=None):
    if prompt_fn is None:
        prompt_fn = input
    if popen_factory is None:
        popen_factory = subprocess.Popen
    if PACKAGE_WORKFLOW_ID != WORKFLOW_ID or SYNTHETIC_SUITE_WORKFLOW_ID != WORKFLOW_ID or DRY_RUN_WORKFLOW_ID != WORKFLOW_ID:
        raise RuntimeError("Run all preparation cells in order; current package/test/dry-run evidence is absent or stale.")
    if DRY_RUN_RESULT.get("status") != "preflight_passed_no_training":
        raise RuntimeError("Current workflow does not have a passing no-training preflight.")
    if not (SYNTHETIC_SUITE_RESULT.get("exit_code") == 0 and PACKAGE_VERSIONS == PINNED):
        raise RuntimeError("Current workflow lacks passing synthetic tests or exact dependency pins.")
    if OUTPUT_ROOT.exists() or RUN_DIR.exists():
        raise FileExistsError(f"Refusing existing run namespace; no overwrite/resume: {RUN_DIR}")
    attempt_marker = SESSION_ROOT / "seed42_topology_attempt_consumed.json"
    if attempt_marker.exists() or attempt_marker.is_symlink():
        raise FileExistsError(f"A seed-42 authorization attempt was already recorded; no retry/resume: {attempt_marker}")
    confirmation = prompt_fn(f"Type {AUTHORIZATION_PHRASE!r} to authorize exactly one run: ").strip()
    if confirmation != AUTHORIZATION_PHRASE:
        print("Authorization not given; no training process was started.")
        return {"status": "not_authorized"}
    launcher = _safe_components(REPOSITORY_ROOT / "phase9_geant/colab_h96_v1/launch.py")
    command = [
        sys.executable, "-I", "-S", str(launcher), "--run-training", "--authorize-training",
        "--repository-root", str(REPOSITORY_ROOT),
        "--source-bundle", str(STAGED_FILES["geant_h96_colab_source_v1.zip"]),
        "--source-manifest", str(STAGED_FILES["source_snapshot_manifest.json"]),
        "--expected-source-bundle-sha256", EXPECTED_FILES["geant_h96_colab_source_v1.zip"],
        "--expected-source-manifest-sha256", EXPECTED_FILES["source_snapshot_manifest.json"],
        "--dataset-bundle", str(STAGED_FILES["geant_h96_colab_trainval_v1.zip"]),
        "--dataset-manifest", str(STAGED_FILES["dataset_bundle_manifest.json"]),
        "--expected-dataset-bundle-sha256", EXPECTED_FILES["geant_h96_colab_trainval_v1.zip"],
        "--expected-dataset-manifest-sha256", EXPECTED_FILES["dataset_bundle_manifest.json"],
        "--seed", "42", "--adjacency-mode", "topology",
    ]
    marker_payload = json.dumps({"workflow_id": WORKFLOW_ID, "command": command, "authorized_at_utc": __import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat()}, sort_keys=True).encode("utf-8")
    with attempt_marker.open("xb") as marker_stream:
        marker_stream.write(marker_payload)
        marker_stream.flush()
        os.fsync(marker_stream.fileno())
    print("Launching the fixed CPU seed-42 topology attempt:", " ".join(command), flush=True)
    process = None
    log_path = SESSION_ROOT / f"seed42_topology_launcher_{uuid.uuid4().hex}.log"
    with log_path.open("x", encoding="utf-8"):
        pass
    def _persist_launcher_line(value):
        with log_path.open("a", encoding="utf-8", newline="") as stream:
            stream.write(value)
            stream.flush()
            os.fsync(stream.fileno())
    exit_code = None
    interrupted = False
    launch_error = None
    try:
        process = popen_factory(command, cwd="/content", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        if process.stdout is None:
            raise RuntimeError("Training launcher stdout pipe was not available")
        for line in process.stdout:
            _persist_launcher_line(line)
            print(line, end="", flush=True)
        exit_code = process.wait()
    except KeyboardInterrupt:
        interrupted = True
        if process is not None and process.poll() is None:
            process.send_signal(signal.SIGINT)
            try:
                exit_code = process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.terminate()
                try:
                    exit_code = process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    exit_code = process.wait()
        print("Interrupted; no automatic retry or resume will be attempted.", flush=True)
    except Exception as exc:
        launch_error = f"{type(exc).__name__}: {exc}"
        print("Launcher failure:", launch_error, flush=True)
        traceback.print_exc()
        _persist_launcher_line("\nLAUNCH ERROR: " + launch_error + "\n")
        if process is not None and process.poll() is None:
            process.send_signal(signal.SIGINT)
            try:
                exit_code = process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.terminate()
                try:
                    exit_code = process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    exit_code = process.wait()
    print("Launcher exit code:", exit_code)
    run_status = "missing"
    manifest_path = RUN_DIR / "run.json"
    verification_error = None
    if manifest_path.is_file():
        try:
            run_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            run_status = run_manifest.get("status", "unknown")
            print("Run lifecycle status:", run_status)
            if run_status == "complete":
                config = run_manifest.get("configuration", {})
                expected_config = {
                    "protocol_id": "geant-horizon96-only-candidate-v2-batch4",
                    "history_length": 672, "horizons": [96], "batch_size": 4,
                    "epochs": 20, "patience": 4, "learning_rate": 0.005,
                    "optimizer": "Adam", "hidden_size": 16, "ewma_alpha": 0.8,
                    "ewma_steps": 5,
                    "residual_loss": "SmoothL1(beta=0.001) in original utilization-ratio units",
                }
                mismatches = {key: (value, config.get(key)) for key, value in expected_config.items() if config.get(key) != value}
                if mismatches:
                    raise RuntimeError(f"Completed run configuration differs from approved settings: {mismatches}")
                if (run_manifest.get("seed") != 42 or run_manifest.get("adjacency_mode") != "topology"
                        or run_manifest.get("checkpoint") != "best.pt"
                        or run_manifest.get("checkpoint_reload_strict") is not True
                        or run_manifest.get("test_access", {}).get("permitted") is not False):
                    raise RuntimeError("Completed run identity, strict reload, or test-access assertion mismatch")
                expected_bundle_identity = {
                    "source_snapshot_manifest_sha256": EXPECTED_FILES["source_snapshot_manifest.json"],
                    "source_bundle_sha256": EXPECTED_FILES["geant_h96_colab_source_v1.zip"],
                    "dataset_manifest_sha256": EXPECTED_FILES["dataset_bundle_manifest.json"],
                    "dataset_bundle_sha256": EXPECTED_FILES["geant_h96_colab_trainval_v1.zip"],
                }
                if config.get("colab_bundle_identity") != expected_bundle_identity:
                    raise RuntimeError("Run manifest bundle identity differs from the approved source/dataset packages")
                expected_resource = config.get("linux_resource_policy", {})
                if (expected_resource.get("policy_id") != "colab-linux-process-memory-v3-visible-cgroup"
                        or expected_resource.get("process_peak_limit_bytes") != 2 * 1024**3):
                    raise RuntimeError("Run manifest does not record the approved v3 2 GiB resource policy")
                if (expected_resource.get("in_run_os_available_reserve_bytes") != 384 * 1024**2
                        or expected_resource.get("in_run_cgroup_headroom_reserve_bytes") != 384 * 1024**2
                        or expected_resource.get("minimum_free_disk_bytes") != 512 * 1024**2):
                    raise RuntimeError("Run manifest memory/disk reserves differ from the approved v3 policy")
                checkpoint = RUN_DIR / "best.pt"
                if not checkpoint.is_file() or checkpoint.stat().st_size <= 0 or _sha256_file(checkpoint) != run_manifest.get("checkpoint_sha256"):
                    raise RuntimeError("Checkpoint is missing, empty, or SHA-256 mismatched")
                prediction_path = RUN_DIR / "validation_predictions.npy"
                if (run_manifest.get("validation_predictions") != "validation_predictions.npy"
                        or not prediction_path.is_file() or prediction_path.stat().st_size <= 0
                        or _sha256_file(prediction_path) != run_manifest.get("validation_predictions_sha256")):
                    raise RuntimeError("Validation prediction artifact is missing, empty, or SHA-256 mismatched")
                if {item.name for item in RUN_DIR.iterdir()} != {"run.json", "best.pt", "validation_predictions.npy"}:
                    raise RuntimeError("Completed run artifact inventory is not exact")
                if not isinstance(run_manifest.get("epoch_history"), list) or not run_manifest["epoch_history"]:
                    raise RuntimeError("Completed run has no epoch history")
                if not isinstance(run_manifest.get("best_epoch"), int) or not (1 <= run_manifest["best_epoch"] <= 20):
                    raise RuntimeError("Completed run has an invalid best epoch")
                if not isinstance(run_manifest.get("runtime_seconds"), (int, float)) or not math.isfinite(run_manifest["runtime_seconds"]):
                    raise RuntimeError("Completed run has an invalid runtime")
                print("Verified checkpoint SHA-256:", run_manifest["checkpoint_sha256"])
                print("Best epoch:", run_manifest.get("best_epoch"), "epochs:", run_manifest.get("epochs_completed"))
                print("Best validation objective:", run_manifest.get("best_validation_h96_residual_smooth_l1"))
                print("Epoch history:", json.dumps(run_manifest.get("epoch_history"), sort_keys=True))
                print("Runtime seconds:", run_manifest.get("runtime_seconds"))
                print("Resource monitor:", json.dumps(run_manifest.get("memory_monitor"), sort_keys=True))
                metrics = run_manifest.get("validation_metrics", {})
                candidate_metrics = metrics.get("h96_candidate", {})
                ewma_metrics = metrics.get("ewma_alpha_0.8", {})
                for metric_name, values in (("candidate", candidate_metrics), ("EWMA", ewma_metrics)):
                    if any(key not in values or not isinstance(values[key], (int, float)) or not math.isfinite(values[key]) for key in ("mae", "rmse")):
                        raise RuntimeError(f"Completed run has invalid {metric_name} validation MAE/RMSE")
                print("Candidate validation metrics:", json.dumps(candidate_metrics, sort_keys=True))
                print("EWMA validation metrics:", json.dumps(ewma_metrics, sort_keys=True))
                if candidate_metrics and ewma_metrics:
                    print("Candidate minus EWMA (validation only):", {
                        "mae": candidate_metrics["mae"] - ewma_metrics["mae"],
                        "rmse": candidate_metrics["rmse"] - ewma_metrics["rmse"],
                    })
        except Exception as exc:
            verification_error = f"{type(exc).__name__}: {exc}"
            print(f"Run verification error: {type(exc).__name__}: {exc}")
            traceback.print_exc()
    else:
        print(f"No run manifest at {manifest_path}; preserve the output namespace and do not retry.")
    effective_status = run_status
    if run_status == "complete" and (exit_code != 0 or verification_error):
        effective_status = "complete_unverified"
    try:
        export = _archive_attempt(RUN_DIR, log_path, SESSION_ROOT, download_fn=download_fn)
        export_error = None
    except Exception as exc:
        export = None
        export_error = f"{type(exc).__name__}: {exc}"
        print(f"Attempt export failed; preserve the run and log in this session: {export_error}")
        traceback.print_exc()
    result = {"status": effective_status, "manifest_status": run_status, "exit_code": exit_code, "interrupted": interrupted,
              "launch_error": launch_error, "verification_error": verification_error, "export_error": export_error,
              "manifest": str(manifest_path), "export": export}
    print("Attempt summary:", json.dumps(result, sort_keys=True))
    return result

TRAINING_RESULT = _authorized_training_workflow()

## Automatic attempt export

The authorization cell above automatically packages the new run directory and launcher log after the single attempt, whether complete or failed. It downloads only that uniquely named archive; no source or dataset package is exported.

In [ ]:
print("No additional manual export cell is needed; TRAINING_RESULT contains the attempt and download summary.")